# Candidate Recommendation — TF-IDF Baseline
Query: **Job description → Top-K candidate resumes**

Datasets (relative paths):
- `./datasets/it_jobs.xlsx`
- `./datasets/UpdatedResumeDataSet.csv`

Outputs:
- `./outputs/candrec_tfidf/baseline_topk.csv`


In [1]:
import os, re, math
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import joblib

DATA_DIR = "./datasets"
OUT_DIR = "./outputs/candrec_tfidf"
os.makedirs(OUT_DIR, exist_ok=True)

JOBS_PATH = os.path.join(DATA_DIR, "/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/datasets/it_jobs.xlsx")
RESUMES_PATH = os.path.join(DATA_DIR, "/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/datasets/UpdatedResumeDataSet.csv")

jobs = pd.read_excel(JOBS_PATH)
resumes = pd.read_csv(RESUMES_PATH)

print("jobs:", jobs.shape, "| resumes:", resumes.shape)
print("job columns:", list(jobs.columns)[:10])
print("resume columns:", list(resumes.columns))


jobs: (21961, 35) | resumes: (962, 2)
job columns: ['#', 'Unnamed: 0', 'id', 'site', 'job_url', 'job_url_direct', 'title', 'company', 'location', 'job_type']
resume columns: ['Category', 'Resume']


In [2]:
def clean_text(s: str) -> str:
    s = str(s or "")
    s = s.lower()
    s = re.sub(r"http\S+|www\S+", " ", s)
    s = re.sub(r"[^a-z0-9\+\#\.\s]", " ", s)  # keep c++, c#, .net
    s = re.sub(r"\s+", " ", s).strip()
    return s


## Build TF-IDF matrix on resume corpus (cached)
This is the expensive step. We cache vectorizer + matrix so multiple queries are fast.


In [3]:
from scipy import sparse

VEC_PATH = os.path.join(OUT_DIR, "tfidf_vectorizer.joblib")
X_PATH = os.path.join(OUT_DIR, "tfidf_resumes.npz")

resume_texts = resumes["Resume"].fillna("").apply(clean_text)

if os.path.exists(VEC_PATH) and os.path.exists(X_PATH):
    vectorizer = joblib.load(VEC_PATH)
    X_resumes = sparse.load_npz(X_PATH)
    print("Loaded cached TF-IDF:", X_resumes.shape)
else:
    vectorizer = TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_features=120_000
    )
    X_resumes = vectorizer.fit_transform(resume_texts)
    joblib.dump(vectorizer, VEC_PATH)
    sparse.save_npz(X_PATH, X_resumes)
    print("Built + cached TF-IDF:", X_resumes.shape)


Loaded cached TF-IDF: (962, 44274)


## Recommend candidates for one job


In [4]:
def build_job_query(job_row: pd.Series) -> str:
    title = str(job_row.get("title", "") or "")
    desc = str(job_row.get("cleaned_description", job_row.get("description", "")) or "")
    return clean_text(title + " " + desc)

def recommend_candidates_tfidf(job_index: int, top_k: int = 10):
    job_row = jobs.iloc[job_index]
    query = build_job_query(job_row)
    q_vec = vectorizer.transform([query])
    sims = cosine_similarity(q_vec, X_resumes).ravel()
    top_idx = np.argsort(-sims)[:top_k]
    out = resumes.iloc[top_idx][["Category"]].copy()
    out.insert(0, "resume_index", top_idx)
    out.insert(0, "job_index", job_index)
    out.insert(2, "job_title", job_row.get("title", ""))
    out["score"] = sims[top_idx]
    return out.reset_index(drop=True)

recommend_candidates_tfidf(0, top_k=10)


,job_index,resume_index,job_title,Category,score
0,0,511,Support Technologist II 2024-02216,Operations Manager,0.424973
1,0,547,Support Technologist II 2024-02216,Operations Manager,0.424973
2,0,515,Support Technologist II 2024-02216,Operations Manager,0.424973
3,0,543,Support Technologist II 2024-02216,Operations Manager,0.424973
4,0,539,Support Technologist II 2024-02216,Operations Manager,0.424973
5,0,519,Support Technologist II 2024-02216,Operations Manager,0.424973
6,0,523,Support Technologist II 2024-02216,Operations Manager,0.424973
7,0,527,Support Technologist II 2024-02216,Operations Manager,0.424973
8,0,531,Support Technologist II 2024-02216,Operations Manager,0.424973
9,0,535,Support Technologist II 2024-02216,Operations Manager,0.424973


### Diễn giải Candidate–Job Similarity Score

| Khoảng điểm | Đánh giá | Diễn giải |
|------------|----------|-----------|
| < 0.30 | Tệ | Ứng viên hầu như không phù hợp với mô tả công việc |
| 0.30 – 0.40 | Yếu | Có một số từ khoá chung nhưng thiếu sự phù hợp tổng thể |
| **0.40 – 0.50** | **Ổn (baseline)** | Ứng viên có mức độ phù hợp chấp nhận được |
| **0.50 – 0.60** | **Tốt** | Ứng viên phù hợp rõ ràng với yêu cầu công việc |
| > 0.60 | Rất tốt (hiếm) | Mức độ phù hợp rất cao, thường chỉ gặp khi nội dung rất tương đồng |


## Run batch experiment and export


In [5]:
job_indices = [0, 10, 50]  # change as needed
all_rows = []
for j in job_indices:
    all_rows.append(recommend_candidates_tfidf(j, top_k=10))
df_out = pd.concat(all_rows, ignore_index=True)

csv_path = os.path.join(OUT_DIR, "baseline_topk.csv")
df_out.to_csv(csv_path, index=False)
csv_path, df_out.head()


('./outputs/candrec_tfidf/baseline_topk.csv',
    job_index  resume_index                           job_title  \
 0          0           511  Support Technologist II 2024-02216   
 1          0           547  Support Technologist II 2024-02216   
 2          0           515  Support Technologist II 2024-02216   
 3          0           543  Support Technologist II 2024-02216   
 4          0           539  Support Technologist II 2024-02216   
 
              Category     score  
 0  Operations Manager  0.424973  
 1  Operations Manager  0.424973  
 2  Operations Manager  0.424973  
 3  Operations Manager  0.424973  
 4  Operations Manager  0.424973  )